# Part C — Machine Learning: Predicting Late Deliveries
**Goal:** flag line items likely to arrive late *at the time the delivery is scheduled*, so the team can intervene early.

**Task:** binary classification, target `is_late` (delivered after the scheduled date, 11.5% of rows).

**Leakage rule (from the EDA objectives):** only fields known when the order is planned are used. Anything computed from the actual delivery date (`delivery_delay_days`, `po_to_delivered_days`, `recording_lag_days`, `flag_severe_late`, `flag_very_early`, `flag_negative_po_lead`, `delivery_performance`) is excluded, as are identifiers.

**Run after** the main notebook, which writes `data/processed/SCMS_Delivery_History_Cleaned.csv`.

In [ ]:
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, PrecisionRecallDisplay, RocCurveDisplay,
                             average_precision_score, f1_score, precision_score,
                             recall_score, roc_auc_score)
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
SEED = 42
sns.set_theme(style="whitegrid", context="notebook")

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CLEAN_PATH = ROOT / "data" / "processed" / "SCMS_Delivery_History_Cleaned.csv"
FIG_DIR, MODEL_DIR = ROOT / "visualizations", ROOT / "models"
for p in (FIG_DIR, MODEL_DIR):
    p.mkdir(parents=True, exist_ok=True)

def save_fig(fig, name):
    fig.savefig(FIG_DIR / f"{name}.png", dpi=200, bbox_inches="tight")
    plt.show(); plt.close(fig)

## 1. Load data and build leakage-free features

In [ ]:
df = pd.read_csv(CLEAN_PATH, parse_dates=["scheduled_delivery_date"])
df = df.dropna(subset=["scheduled_delivery_date", "delivery_delay_days"]).sort_values("scheduled_delivery_date").reset_index(drop=True)
df["scheduled_month"] = df["scheduled_delivery_date"].dt.month
df["scheduled_year"] = df["scheduled_delivery_date"].dt.year

LOG_COLS = ["line_item_quantity", "line_item_value", "unit_price", "pack_price", "weight_kilograms", "freight_cost_usd"]
NUM_COLS = LOG_COLS + ["po_to_scheduled_days", "scheduled_year", "scheduled_month"]
CAT_COLS = ["fulfill_via", "shipment_mode", "vendor_inco_term", "country", "managed_by", "product_group",
            "sub_classification", "vendor", "manufacturing_site", "first_line_designation", "dosage_form",
            "molecule_test_type", "scheduled_dow", "weight_data_status", "freight_data_status",
            "po_date_status", "pq_date_status"]
FEATURES = NUM_COLS + CAT_COLS
LEAKY = ["delivery_delay_days", "delivery_performance", "po_to_delivered_days", "recording_lag_days",
         "flag_severe_late", "flag_very_early", "flag_negative_po_lead", "delivered_to_client_date",
         "delivery_recorded_date", "is_late"]
assert not set(FEATURES) & set(LEAKY), "leakage: a post-delivery column is in the feature list"

X, y = df[FEATURES], df["is_late"]
print(f"{len(df):,} rows | {len(FEATURES)} features | late rate {y.mean():.1%}")

## 2. Train/test design
Two splits are used because the late rate jumped from 2.4% (2007–2009) to 15.2% (2010 onwards):
- **Random stratified split (80/20)** — how well the model separates late from not-late overall.
- **Time-based split** — train on the earliest 80% of schedules, test on the latest 20%. This is the realistic test, because in production the model always predicts the future.

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
cut = int(len(df) * 0.8)
Xtr_t, Xte_t, ytr_t, yte_t = X.iloc[:cut], X.iloc[cut:], y.iloc[:cut], y.iloc[cut:]
print(f"Random  : train {len(Xtr):,} | test {len(Xte):,} | test late rate {yte.mean():.1%}")
print(f"Temporal: train {len(Xtr_t):,} | test {len(Xte_t):,} | test late rate {yte_t.mean():.1%} "
      f"({df['scheduled_delivery_date'].iloc[cut].date()} onwards)")

## 3. Preprocessing and candidate models

In [ ]:
def build_preprocessor(scale):
    num_steps = [SimpleImputer(strategy="median", add_indicator=True)]
    if scale:
        num_steps = [SimpleImputer(strategy="median", add_indicator=True), StandardScaler()]
    log_pipe = make_pipeline(FunctionTransformer(np.log1p, feature_names_out="one-to-one"),
                             *num_steps)
    plain_cols = [c for c in NUM_COLS if c not in LOG_COLS]
    return ColumnTransformer([
        ("log", log_pipe, LOG_COLS),
        ("num", make_pipeline(*num_steps), plain_cols),
        ("cat", make_pipeline(SimpleImputer(strategy="constant", fill_value="Missing"),
                              OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=30, sparse_output=False)),
         CAT_COLS),
    ])

models = {
    "Baseline (always predict rate)": DummyClassifier(strategy="prior"),
    "Logistic Regression": Pipeline([("prep", build_preprocessor(True)),
        ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", C=0.3))]),
    "Random Forest": Pipeline([("prep", build_preprocessor(False)),
        ("clf", RandomForestClassifier(n_estimators=400, min_samples_leaf=3, class_weight="balanced_subsample",
                                       n_jobs=-1, random_state=SEED))]),
    "Gradient Boosting": Pipeline([("prep", build_preprocessor(False)),
        ("clf", HistGradientBoostingClassifier(learning_rate=0.05, max_iter=300, max_leaf_nodes=15,
                                               l2_regularization=1.0, class_weight="balanced", random_state=SEED))]),
}

## 4. Compare models (ROC-AUC and PR-AUC; accuracy is misleading at an 11.5% positive rate)

In [ ]:
def score(model, Xa, ya, Xb, yb):
    model.fit(Xa, ya)
    p = model.predict_proba(Xb)[:, 1]
    return {"ROC-AUC": roc_auc_score(yb, p), "PR-AUC": average_precision_score(yb, p)}

rows = []
for name, m in models.items():
    r = score(m, Xtr, ytr, Xte, yte); t = score(m, Xtr_t, ytr_t, Xte_t, yte_t)
    rows.append({"model": name, "random ROC-AUC": r["ROC-AUC"], "random PR-AUC": r["PR-AUC"],
                 "temporal ROC-AUC": t["ROC-AUC"], "temporal PR-AUC": t["PR-AUC"]})
results = pd.DataFrame(rows).set_index("model").round(3)
display(results)
print(f"PR-AUC of a no-skill model = late rate: random {yte.mean():.3f} | temporal {yte_t.mean():.3f}")

**How to read this:** PR-AUC must be compared with the late rate (a no-skill model scores about that value). The gap between the random and temporal columns shows how much of the skill survives a change in period. Report the temporal numbers as the honest expectation.

## 5. Pick the best model, choose an alert threshold, and evaluate

In [ ]:
best_name = results["temporal PR-AUC"].drop("Baseline (always predict rate)").idxmax()
best = models[best_name]
print("Best model (by temporal PR-AUC):", best_name)

# Threshold chosen on out-of-fold predictions of the temporal TRAIN set only (no peeking at the test set).
oof = cross_val_predict(best, Xtr_t, ytr_t, cv=StratifiedKFold(5, shuffle=True, random_state=SEED), method="predict_proba")[:, 1]
grid = np.linspace(0.05, 0.95, 91)
thr = grid[np.argmax([f1_score(ytr_t, oof >= t) for t in grid])]

best.fit(Xtr_t, ytr_t)
proba = best.predict_proba(Xte_t)[:, 1]
pred = (proba >= thr).astype(int)
print(f"Threshold {thr:.2f} | precision {precision_score(yte_t, pred):.2f} | recall {recall_score(yte_t, pred):.2f} "
      f"| F1 {f1_score(yte_t, pred):.2f} | ROC-AUC {roc_auc_score(yte_t, proba):.2f} | PR-AUC {average_precision_score(yte_t, proba):.2f}")

fig, ax = plt.subplots(1, 3, figsize=(17, 4.8))
RocCurveDisplay.from_predictions(yte_t, proba, ax=ax[0], name=best_name); ax[0].plot([0, 1], [0, 1], "k--", lw=1)
ax[0].set_title("ROC Curve (time-based test set)")
PrecisionRecallDisplay.from_predictions(yte_t, proba, ax=ax[1], name=best_name)
ax[1].axhline(yte_t.mean(), color="k", ls="--", lw=1, label="No-skill"); ax[1].legend(); ax[1].set_title("Precision-Recall Curve")
ConfusionMatrixDisplay.from_predictions(yte_t, pred, display_labels=["Not late", "Late"], cmap="Blues", ax=ax[2], colorbar=False)
ax[2].set_title(f"Confusion Matrix (threshold {thr:.2f})"); ax[2].grid(False)
fig.tight_layout(); save_fig(fig, "27_model_evaluation")

**Why this chart:** ROC shows overall ranking ability, the precision-recall curve shows the trade-off that matters when late items are rare, and the confusion matrix shows the real counts of caught and missed late deliveries.

**Insight / Business impact:** read the numbers printed above. Write them into the project summary as: "At threshold X the model catches R% of late deliveries with P% precision."

## 6. Business view: how many late items do we catch by reviewing only the riskiest shipments?

In [ ]:
gain = pd.DataFrame({"y": yte_t.values, "p": proba}).sort_values("p", ascending=False).reset_index(drop=True)
gain["share_reviewed"] = (gain.index + 1) / len(gain)
gain["late_caught"] = gain["y"].cumsum() / gain["y"].sum()

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.plot(gain["share_reviewed"], gain["late_caught"], color="#E15759", lw=2.5, label=best_name)
ax.plot([0, 1], [0, 1], "k--", lw=1, label="Random review")
ax.set(title="Cumulative Gain: Late Deliveries Caught vs Shipments Reviewed", xlabel="Share of shipments reviewed (highest risk first)",
       ylabel="Share of late deliveries caught")
ax.xaxis.set_major_formatter(lambda v, _: f"{v:.0%}"); ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}"); ax.legend()
save_fig(fig, "28_cumulative_gain")

for s in (0.10, 0.20, 0.30):
    caught = gain.loc[gain["share_reviewed"] <= s, "y"].sum() / gain["y"].sum()
    print(f"Reviewing the riskiest {s:.0%} of shipments catches {caught:.0%} of late deliveries (random review: {s:.0%})")

**Why this chart:** it answers the operational question directly: if the team can only check a limited share of shipments, how many late ones does the model find compared with picking at random?

**Business impact:** proactive expediting on the top-risk 10–20% of orders, in place of blanket monitoring, focuses limited staff time where lateness is most likely.

## 7. What drives the predictions? (permutation importance on the test set)

In [ ]:
imp = permutation_importance(best, Xte_t, yte_t, scoring="average_precision", n_repeats=10, random_state=SEED, n_jobs=-1)
imp_df = pd.Series(imp.importances_mean, index=FEATURES).sort_values().tail(12)
err = pd.Series(imp.importances_std, index=FEATURES)[imp_df.index]

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.barh(imp_df.index.str.replace("_", " "), imp_df.values, xerr=err.values, color="#4C78A8")
ax.set(title="Top Drivers of Late-Delivery Prediction (Permutation Importance)", xlabel="Drop in PR-AUC when the feature is shuffled")
fig.tight_layout(); save_fig(fig, "29_feature_importance")
print(imp_df.sort_values(ascending=False).round(4).to_string())

**Why this chart:** permutation importance measures how much accuracy is lost when one feature is scrambled, so it is model-agnostic and works with one-hot columns grouped back to the original field.

**Insight:** compare this ranking with the EDA. The EDA found route (`fulfill_via`), vendor, country/lane and period matter, while quantity, value and weight are not linked to delay. Agreement between the two is a useful consistency check; any disagreement is worth a sentence in the write-up.

## 8. Save the model and score new shipments

In [ ]:
final = best.fit(X, y)                                   # refit on all data for deployment
joblib.dump({"model": final, "features": FEATURES, "threshold": float(thr)}, MODEL_DIR / "late_delivery_model.joblib")

def flag_high_risk(new_df, path=MODEL_DIR / "late_delivery_model.joblib"):
    bundle = joblib.load(path)
    out = new_df.copy()
    out["late_risk"] = bundle["model"].predict_proba(new_df[bundle["features"]])[:, 1]
    out["flag_high_risk"] = (out["late_risk"] >= bundle["threshold"]).astype(int)
    return out.sort_values("late_risk", ascending=False)

flag_high_risk(X.tail(500)).head(10)[["fulfill_via", "shipment_mode", "country", "vendor", "late_risk", "flag_high_risk"]]

## 9. Model limitations
- **Time drift:** the late rate rose sharply from 2010, so performance on future periods is best estimated by the time-based split, not the random one. Retrain regularly.
- **Scheduled-date target:** `is_late` is measured against the promised date. Padded schedules inflate early deliveries and hide true lateness.
- **Data gaps:** freight, weight and PO date are missing for 40–55% of rows; the model uses missing-status flags, which also reflect how the record was created.
- **Associations, not causes:** importance shows what predicts lateness, not what causes it. Use the EDA recommendations (RDC route, three vendors, worst lanes) for root-cause action.
- **Freight and weight timing:** if these are only known after dispatch, drop them and re-run for a strictly order-time model.